# Pricing Data Processing

### Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

### Schema setup for Bronze, Silver and Gold layers

In [0]:
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

### Defining widgets for runtime configuration
#### Syntax: dbutils.widgets.text(internal handle, pre-filled value, display text)
#### /*.csv = 'read every csv file in this folder'

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog)
print(data_source)

base_path = f's3://databricks-data-engineering-project/{data_source}/*.csv'
print(base_path)

fmcg
gross_price
s3://databricks-data-engineering-project/gross_price/*.csv


### Read CSV from source

In [0]:
df = ( 
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

display(df.limit(10))

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-10-23T12:44:28.198Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-10-23T12:44:28.198Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-10-23T12:44:28.198Z,gross_price.csv,2741


### Inspect the inferred schema

In [0]:
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### Ingesting raw data into Bronze layer

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
)

### Silver Processing

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')
df_bronze.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025/07/01|        -84|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|01/08/2025|    unknown|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|2025/09/01|         84|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|2025-10-01|         83|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|2025-11-01|         83|2026-10-23 12:44:...|gross_price.csv|     2741|
|  88888888|2025-12-01|        -83|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891102|2025-07-01|         68|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891102|2025-08-01|         68|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891102|2025-09-01|         68|2026-10-23 12:44:...|gross_price.csv|     2741|
|  2

### Standardize `month` column

In [0]:
display(spark.sql(f"""
SELECT
    month
FROM
    {catalog}.{bronze_schema}.{data_source};
"""))

month
2025/07/01
01/08/2025
2025/09/01
2025-10-01
2025-11-01
2025-12-01
2025-07-01
2025-08-01
2025-09-01
2025-10-01


### Parse `month` from multiple possible formats

In [0]:
date_formats = ['yyyy/MM/dd', 'dd/MM/yyyy', 'yyyy-MM-dd', 'dd-MM-yyyy']

df_silver = df_bronze.withColumn('month',
        coalesce(
        try_to_date(col('month'), 'yyyy/MM/dd'),
        try_to_date(col('month'), 'dd/MM/yyyy'),
        try_to_date(col('month'), 'yyyy-MM-dd'),
        try_to_date(col('month'), 'dd-MM-yyyy')))

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify dates in `month` column

In [0]:
display(spark.sql(f"""
SELECT
    month
FROM
    {catalog}.{silver_schema}.{data_source}
LIMIT 10;
"""))

month
2025-07-01
2025-08-01
2025-09-01
2025-10-01
2025-11-01
2025-12-01
2025-07-01
2025-08-01
2025-09-01
2025-10-01


### Handle `gross_price` column

In [0]:
display(spark.sql(f"""
SELECT
    gross_price
FROM
    {catalog}.{silver_schema}.{data_source}
LIMIT 10;
"""))

gross_price
-84
unknown
84
83
83
-83
68
68
68
69


### Clean values in `gross_price` column

In [0]:
df_silver = (
    df_silver.withColumn('gross_price', 
    when(col('gross_price').rlike(r'-?[0-9]+(\.[0-9]+)?'), 
    abs(col('gross_price')).cast('double'))
    .otherwise(lit(0.0)))
)

# -? - Optional minus sign
# (\.[0-9]+)? - Optional group

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify cleaned values

In [0]:
display(spark.sql(f"""
SELECT
    gross_price
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

gross_price
84.0
0.0
84.0
83.0
83.0
83.0
68.0
68.0
68.0
69.0


#### Enrich the silver dataset by performing an inner join with the `products` table to retrieve the `product_code` for each `product_id`

In [0]:
df_products = spark.table(f'{catalog}.{silver_schema}.products')

df_joined = (
    df_silver.join(df_products.select('product_id', 'product_code'), 
    on = 'product_id', how = 'inner')
)

df_joined = df_joined.select('product_id', 'product_code', 'month', 'gross_price', 'read_timestamp', 'file_name', 'file_size')

df_joined.show()

+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|product_id|        product_code|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|  25891101|e91ba9d665f90254d...|2025-07-01|       84.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-08-01|        0.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-09-01|       84.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-10-01|       83.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-11-01|       83.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891102|e92c739a8d78cd6cb...|2025-07-01|       68.0|2026-10-23 12:44:...|gross_price.csv|     2741|
|  25891102|e92c739a8d78cd6cb...|2025-08-01|       68.0|2026-10-

### Write Transformed Data to Silver

In [0]:
(
    df_joined.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Gold Processing - Select final columns

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source}')
df_gold = df_silver.select('product_code', 'month', 'gross_price')

### Writing staging dimension to gold schema

In [0]:
(
    df_gold.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{gold_schema}.staging_dim_{data_source}')
)

### Latest Non-Zero Price per Product per Year (SQL)

In [0]:
display(spark.sql(f"""
WITH latest_price AS
(SELECT
    product_code,
    month,
    gross_price,
    ROW_NUMBER() OVER(PARTITION BY product_code, YEAR(month) ORDER BY month DESC)
    AS rn
FROM
    {catalog}.{gold_schema}.staging_dim_{data_source}
WHERE
    gross_price <> 0)

SELECT
    product_code,
    YEAR(month) AS year,
    gross_price
FROM
    latest_price
WHERE
    rn = 1;
"""))

product_code,year,gross_price
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,281.0
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,2025,100.0
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,86.0
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,2025,108.0
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,493.0
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,187.0
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,440.0
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,296.0
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,50.0
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,2025,138.0


### Latest Non-Zero Price per Product per Year (PySpark)

In [0]:
df_gold_latest_price = spark.table(f'{catalog}.{gold_schema}.staging_dim_{data_source}')

df_gold_latest_price = (
    df_gold_latest_price
    .withColumn('year', year(col('month')))
    .filter(col('gross_price') != 0)
    .withColumn('row_num', row_number().over(
    Window.partitionBy(col('product_code'), col('year'))
    .orderBy(col('month').desc()) 
    ))
    .filter(col('row_num') == 1)
    .drop('row_num')
    .withColumnRenamed('gross_price', 'price_inr')
    .drop('month')
)

display(df_gold_latest_price)

product_code,price_inr,year
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,281.0,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,100.0,2025
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,86.0,2025
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,108.0,2025
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,493.0,2025
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,187.0,2025
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,440.0,2025
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,296.0,2025
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,50.0,2025
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,138.0,2025


### Stage `df_gold_latest_price` as TempView for MERGE

In [0]:
df_gold_latest_price.createOrReplaceTempView('latest_price_src')

### Merge TempView with parent

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.dim_{data_source} AS target
USING latest_price_src AS source
ON
    target.product_code = source.product_code

WHEN MATCHED THEN
UPDATE
SET
    price_inr = source.price_inr,
    year = source.year

WHEN NOT MATCHED THEN
INSERT (
    product_code,
    price_inr,
    year
)
VALUES (
    source.product_code,
    source.price_inr,
    source.year  
);

"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
17,0,0,17
